# Trade Decision Lab | Reproducible research demonstration

**Research:** Garbage In, Garbage Out: Examining the Implications of Poor Data Quality for Decision-Making in Nigeria's Import–Export Sector.

This notebook runs the preserved UN Comtrade research engine: **Model A** audits quality, **Model B** measures analytical sensitivity, and **Model C** examines potential decision implications. It does not claim to observe actual trader decisions. Run cells in order. The project ZIP and three compatible trade files are the only required uploads; the LLM is optional.
This release uses `openpyxl` for Excel reporting and installs all dependencies from public PyPI.


## 1. Project architecture

The uploaded project ZIP contains `trade_decision_lab/comtrade_engine/` with the original quality, comparison, decision, schema, config and pipeline modules. `integration.py` provides the stable application entry point. `reporting.py` exports engine-produced evidence. `llm.py` is a separate OpenAI-compatible interpretation client. The notebook and Streamlit application do **not** reimplement Models A–C. The general-purpose `engine.py` is a distinct external-dataset workflow and is not used for this empirical study.

## 2. Set up the environment

Upload the **completed project ZIP** when prompted. The extraction cell searches for the project root instead of assuming a particular Google Drive location. Optional: mount Drive separately if you wish to copy files there.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, sys, subprocess, os
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1: raise ValueError('Upload exactly one project ZIP in this cell.')
if Path(zip_names[0]).name != 'Trade_Decision_Lab_Project_FINAL.zip':
    raise ValueError('Upload Trade_Decision_Lab_Project_FINAL.zip supplied with this notebook, not an older project archive.')
extract_dir = Path('/content/tdl_project')
if extract_dir.exists():
    import shutil
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True)
with zipfile.ZipFile(zip_names[0]) as archive:
    for member in archive.infolist():
        destination = (extract_dir / member.filename).resolve()
        if not destination.is_relative_to(extract_dir.resolve()): raise ValueError('Unsafe ZIP member path')
    archive.extractall(extract_dir)
roots = [q.parent for q in extract_dir.rglob('requirements.txt') if (q.parent/'trade_decision_lab'/'integration.py').exists()]
if len(roots) != 1: raise RuntimeError('Could not uniquely locate the completed Trade Decision Lab package.')
PROJECT_DIR=roots[0]
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(PROJECT_DIR/'requirements.txt')])
sys.path.insert(0,str(PROJECT_DIR))
from trade_decision_lab.comtrade_engine.decision import _baseline_present
assert _baseline_present('Latvia | Poland | USA') and not _baseline_present('[]'), 'Incorrect baseline-detection version'
print('Ready:', PROJECT_DIR, '| Corrected baseline detection verified')

## 3. Upload and inspect research datasets

Upload exactly three CSV/XLSX files. **Upload order does not matter.** After uploading, choose the correct role for each file using the dropdowns. The following validation cell checks schema and trade-flow direction (`M` = imports, `X` = exports). A total-trade file can contain both flows; a warning appears if only one is present.


In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from trade_decision_lab.integration import load_file, run_analysis
from trade_decision_lab.comtrade_engine.schema import validate_comtrade_frame, SchemaError
uploaded_data = files.upload()
valid_names = [name for name in uploaded_data if name.lower().endswith(('.csv', '.xlsx'))]
print('Uploaded files:')
for i, name in enumerate(valid_names):
    print(f'  {i}: {name}')
if len(valid_names) != 3:
    raise ValueError('Upload exactly three CSV/XLSX files: imports, exports and total trade.')

def suggested_name(role):
    terms = {'Imports': ('import',), 'Exports': ('export',), 'Total trade': ('total', 'comtrade_nigeria')}[role]
    matches = [name for name in valid_names if any(term in name.lower() for term in terms)]
    return matches[0] if len(matches) == 1 else valid_names[0]

selectors = {}
for role in ('Imports', 'Exports', 'Total trade'):
    selectors[role] = widgets.Dropdown(options=valid_names, value=suggested_name(role), description=role + ':', style={'description_width': '110px'}, layout=widgets.Layout(width='700px'))
    display(selectors[role])
print('Review all three selections. Then run the NEXT cell to validate and load the datasets.')


### Confirm the assignments and validate the data

Select the correct files in the dropdowns above **before** running the next cell. If you change a dropdown later, rerun the validation cell and all subsequent analytical cells.


In [ ]:
assignments = {role: selector.value for role, selector in selectors.items()}
print('Confirmed selections:', assignments)
if len(set(assignments.values())) != 3:
    raise ValueError('Each dataset role must use a different file. Change the dropdown selections above and rerun this cell.')
frames = {role: load_file(filename, filename) for role, filename in assignments.items()}
for role, frame in frames.items():
    validation = validate_comtrade_frame(frame, strict=False)
    print(f'\n{role}: {assignments[role]} | {len(frame):,} records | {len(frame.columns)} columns | Schema valid: {validation["valid"]}')
    if not validation['valid']:
        raise SchemaError(f'{role}: {validation}')
    if 'flowCode' not in frame.columns:
        raise ValueError(f'{role}: flowCode is required to verify the assigned trade direction.')
    flows = set(frame['flowCode'].dropna().astype(str).str.strip().str.upper().unique())
    print('  flowCode values:', sorted(flows))
    if role == 'Imports' and (not flows or flows != {'M'}):
        raise ValueError(f'Imports must contain only flowCode M; found {sorted(flows)}. Check the dropdown.')
    if role == 'Exports' and (not flows or flows != {'X'}):
        raise ValueError(f'Exports must contain only flowCode X; found {sorted(flows)}. Check the dropdown.')
    if role == 'Total trade' and (not flows or not flows.issubset({'M','X'})):
        raise ValueError(f'Total trade contains unexpected flowCode values {sorted(flows)}.')
    if role == 'Total trade' and flows != {'M','X'}:
        print('  WARNING: Total trade contains only one trade direction. Confirm the source extract covers the intended scope.')
    display(frame.head(3))
display(pd.DataFrame([{'Dataset': role, 'Filename': assignments[role], 'Records': len(frame), 'Columns': len(frame.columns), 'Flows': ', '.join(sorted(set(frame['flowCode'].dropna().astype(str).str.upper())))} for role,frame in frames.items()]))
print('Dataset assignment and schema checks complete. Review any warnings before proceeding.')


## 4. Execute Models A, B and C

One engine call performs the quality audits, original configured commodity/period comparisons, sensitivity treatments and potential decision assessments. Detailed CSVs, including traceability, are written by the engine.

In [ ]:
from trade_decision_lab.comtrade_engine import config

OUTPUT_DIR = Path('/content/trade_decision_lab_results')

results = run_analysis(
    frames['Imports'],
    frames['Exports'],
    frames['Total trade'],
    OUTPUT_DIR
)

print('Engine complete. Output:', results['output_dir'])
print('Configured commodity codes:', config.SELECTED_COMMODITIES)

## 5. Model A: Data quality assessment

Inspect observed quality conditions separately for imports, exports and the total-trade reference. Screening flags do not establish that a record is erroneous.

In [ ]:
from trade_decision_lab.comtrade_engine.quality import findings_summary
for flow,audit in results['audits'].items():
    print('MODEL A:',flow)
    display(findings_summary(audit))

## 6. Model B: Analytical consequences

Rankings and sensitivity tables are the engine's actual outputs. Alternative treatments may change leading partners, concentration or the eligible records. A changed result is evidence of sensitivity, not proof of a better commercial decision.

In [ ]:
import matplotlib.pyplot as plt
for flow,comparison in results['comparisons'].items():
    print('MODEL B:',flow)
    for label in ('periods','rankings','sensitivity'):
        frame=comparison[label];print(label,len(frame));display(frame.head(30))
    ranks=comparison['rankings']
    if not ranks.empty:
        numeric=[c for c in ['trade_value','value','share_pct'] if c in ranks.columns]
        if numeric and 'partnerDesc' in ranks.columns:
            ranks.head(10).plot.bar(x='partnerDesc',y=numeric[0],title=f'{flow}: engine partner ranking preview');plt.show()

## 7. Model C: Decision-specific fitness for use

Review the three distinct decision assessments and their reasons. This notebook requires the updated project ZIP containing the fitness-for-use rules. If you upload an older ZIP, the check below stops rather than presenting outdated findings.

In [ ]:
from pathlib import Path
import pandas as pd

required_fitness = [
    'partner_prioritisation_fitness', 'concentration_fitness', 'unit_value_fitness',
    'partner_prioritisation_reason', 'concentration_reason', 'unit_value_reason',
    'partner_prioritisation_action', 'concentration_action', 'unit_value_action'
]

# Read the persisted engine evidence, which is also the source for the reports.
decision_frames = {}
for flow in ('Import', 'Export'):
    path = Path(OUTPUT_DIR) / flow.lower() / 'model_c_decision_assessment.csv'
    decisions = pd.read_csv(path, dtype={'commodity_code': str})
    missing = sorted(set(required_fitness) - set(decisions.columns))
    if missing:
        raise RuntimeError(
            f'{flow}: the uploaded project ZIP does not contain the final Model C '
            f'fitness-for-use rules. Missing: {missing}. Repackage the updated '
            'trade_decision_lab project and rerun the notebook from setup.'
        )
    decision_frames[flow] = decisions
    print(f'\n{flow}: {len(decisions)} commodity-period assessments')
    display(decisions[[
        'scenario', 'classification', 'commodity_code', 'commodity_description',
        'records', 'country_partners', *required_fitness[:3]
    ]])

## 8. Prepare a concise, verified context for the optional LLM

The technical PDF and workbook remain the authoritative outputs. The optional assistant receives selected, structured findings and explains their implications in ordinary language. It does not independently calculate results or replace the research evidence.

In [ ]:
import json
from trade_decision_lab.comtrade_engine.quality import findings_summary

context = {
    'scope': 'Nigerian UN Comtrade data, configured HS4 commodities and periods',
    'input_shapes': {name: list(frame.shape) for name, frame in frames.items()},
    'model_a': {},
    'model_c': {},
    'interpretation_rule': 'Sensitivity is not proof of error or observed decision change.'
}
for flow, audit in results['audits'].items():
    context['model_a'][flow] = findings_summary(audit).head(20).to_dict('records')

# One record per unique flow, classification, years and commodity. The configured
# A_historical H6 and B_recent H6 periods overlap and must not be double-counted.
for flow, frame in decision_frames.items():
    distinct = frame.drop_duplicates(
        subset=['classification', 'years', 'commodity_code'], keep='last'
    )
    selected = [
        'classification', 'years', 'commodity_code', 'commodity_description',
        'records', 'country_partners', 'decision_sensitivity',
        'reported_weight_value_share_pct', 'estimated_weight_value_share_pct',
        'missing_weight_value_share_pct',
        'partner_prioritisation_fitness', 'partner_prioritisation_reason',
        'concentration_fitness', 'concentration_reason',
        'unit_value_fitness', 'unit_value_reason'
    ]
    context['model_c'][flow] = distinct[selected].where(pd.notna(distinct[selected]), None).to_dict('records')

# Model B: summarise alternative analytical treatments, excluding the unchanged baseline.
# Diagnostic World-aggregate treatments are counted separately and are not corrections.
context['model_b'] = {}
for flow, comparison in results['comparisons'].items():
    sensitivity = comparison['sensitivity']
    alternatives = sensitivity.loc[sensitivity['treatment'].ne('country_only_baseline')].copy()
    diagnostic = alternatives['treatment'].eq('all_partner_labels_diagnostic')
    valid = alternatives.loc[~diagnostic]
    def changed(frame, field):
        return int(frame[field].astype(str).str.lower().eq('true').sum())
    examples = valid.loc[valid['top1_changed'].astype(str).str.lower().eq('true')].head(2)
    context['model_b'][flow] = {
        'non_diagnostic_comparisons': len(valid),
        'non_diagnostic_leading_partner_changes': changed(valid, 'top1_changed'),
        'non_diagnostic_top_five_changes': changed(valid, 'top5_membership_changed'),
        'diagnostic_comparisons_excluded_from_counts': int(diagnostic.sum()),
        'examples': [{
            'commodity_code': str(row['commodity_code']),
            'commodity': row['commodity_description'],
            'scenario': row['scenario'],
            'treatment': row['treatment'],
            'baseline_partner': row['baseline_top1'],
            'alternative_partner': row['treatment_top1']
        } for _, row in examples.iterrows()]
    }
context['interpretation_rule'] += (
    ' Model B counts are comparisons between treatments, not independent errors or real decisions. '
    'World-aggregate diagnostic treatments are excluded from the displayed change counts.'
)

context_text = json.dumps(context, default=str, ensure_ascii=False)
print('LLM evidence context prepared:', len(context_text), 'characters')

## 9. Optional plain-English interpretation

This is a short explanation for a non-technical reader, not an appendix to the technical report. It is displayed in Colab only. Add `GEM_KEY` in Colab Secrets to enable it; otherwise the deterministic research and downloads work normally.

In [ ]:
from trade_decision_lab.llm import chat
from google.colab import userdata
from IPython.display import display, Markdown

try:
    API_KEY = userdata.get('GEM_KEY')
except Exception:
    API_KEY = None
MODEL = 'gemini-3.5-flash-lite'  # Change if this model is unavailable to your account.
BASE_URL = 'https://generativelanguage.googleapis.com/v1beta/openai'

SYSTEM = (
    'You are a plain-English guide to a Nigerian trade data-quality research demonstration. '
    'Use only the supplied evidence. Do not invent statistics, partners, causal effects or '
    'actual commercial decisions. Explain that flags are not confirmed errors, sensitivity '
    'is not a correction, and the fitness verdicts are preliminary screening judgements. '
    'Use clear British English, short sentences and everyday language. Avoid technical '
    'jargon; explain unavoidable terms briefly. Do not reproduce the technical report. '
    'Make decision-specific differences clear: use with caution is not the same as insufficient evidence. '
    'Whenever giving a changed-partner example, identify the specific treatment and scenario. '
    'Do not imply a diagnostic World-aggregate scenario is a corrected result.'
)

brief = None
if API_KEY:
    try:
        brief = chat(
            API_KEY, MODEL,
            [
                {'role': 'system', 'content': SYSTEM},
                {'role': 'user', 'content': (
                    'Write a concise, accessible summary (180 to 260 words) with four '
                    'short sections: What we examined; What stood out; What this means '
                    'for decisions; What we cannot conclude. Use at most three specific '
                    'examples and only figures supplied in the evidence. Do not enumerate '
                    'all commodities, tests or verdicts. Do not repeat the report. '
                    'Give one concrete Model C contrast from the evidence if available. '
                    'Do not add a download instruction; the notebook appends one consistently. '
                    'Evidence:\n' + context_text
                )}
            ], BASE_URL
        )
        display(Markdown('## Report summary\n\n' + brief.strip() + '\n\n**For the full findings, methods and evidence, download the technical PDF report using the button below.**'))
    except Exception as exc:
        print('Optional LLM summary unavailable:', exc)
        display(Markdown('## Report summary\n\nThe summary could not be generated. Please use the technical report below for the verified findings.'))
else:
    display(Markdown('## Report summary\n\nThe optional plain-English summary is unavailable until a GEM_KEY is configured in Colab Secrets. The research evidence and reports are available below.\n\n**For the full findings, methods and evidence, download the technical PDF report using the button below.**'))

## 10. Ask a follow-up question (optional)

Change `question` and run this cell. Answers are limited to the evidence supplied to the assistant. This does not modify the report.

In [ ]:
question = 'What are the most important limitations of these results?'
if API_KEY:
    try:
        answer = chat(
            API_KEY, MODEL,
            [
                {'role': 'system', 'content': SYSTEM},
                {'role': 'user', 'content': 'Research evidence:\n' + context_text},
                {'role': 'user', 'content': question}
            ], BASE_URL
        )
        display(Markdown(answer))
    except Exception as exc:
        print('Optional follow-up unavailable:', exc)
else:
    print('Configure GEM_KEY to enable follow-up questions.')

## 11. Prepare reports and choose what to download

The PDF and Excel workbook contain **only deterministic engine evidence**, never the LLM summary. Click a button to download a specific file. The complete ZIP includes CSV evidence and the reports. Nothing downloads automatically.

In [ ]:
import importlib
import shutil
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from google.colab import files
from trade_decision_lab import reporting

importlib.reload(reporting)
reports = reporting.generate_reports(OUTPUT_DIR, llm_brief=None)
report_paths = {kind: Path(path) for kind, path in reports.items()}
archive_path = Path(shutil.make_archive(
    '/content/Trade_Decision_Lab_All_Results', 'zip', str(OUTPUT_DIR)
))
report_paths['complete ZIP'] = archive_path

for label, path in report_paths.items():
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(f'Report missing or empty: {path}')

print('Reports are ready. Choose a file to download:')
buttons = []
for kind, path in report_paths.items():
    button = widgets.Button(
        description={'pdf': 'Download PDF report',
                     'xlsx': 'Download Excel workbook',
                     'complete ZIP': 'Download all evidence (ZIP)'}[kind],
        button_style='', layout=widgets.Layout(width='260px')
    )
    def download_on_click(_button, selected_path=path):
        # Native Colab download is invoked only by an explicit user click.
        files.download(str(selected_path))
    button.on_click(download_on_click)
    buttons.append(button)
display(widgets.VBox(buttons))

## 12. Research interpretation and handover

**Evidence hierarchy:** Model A identifies observed data conditions; Model B measures how selected analytical outputs respond to alternative treatments; Model C assigns preliminary, rule-based fitness-for-use judgements for partner prioritisation, market concentration and unit-value comparisons. These are not observed changes in actual procurement or policy decisions.

**Methodological notes:** The five-record and three-partner cut-offs are prototype screening heuristics, not statistically validated universal thresholds. H4 and H6 identify classification revisions, not the number of digits in HS codes. Historical H6 and recent H6 outputs overlap, so do not count them as independent cases. World aggregates must not be added to country partners. The complete CSV evidence and source-row traceability are retained in the downloadable evidence package.

**Reproducibility:** Save the final, updated project ZIP containing the modified `comtrade_engine/decision.py` and `reporting.py` alongside this notebook. A new Colab session loads code from that ZIP; changes made only to a previous Colab runtime will not carry over.